In [ ]:
import numpy as np
import random
import matplotlib.pyplot as plt
from collections import deque
import sys

In [ ]:
class HybridPowerFlowOptimizer:
   
    def __init__(self, A_matrix, line_limits, gen_costs, gen_limits_min, gen_limits_max, initial_B, memory_size=100):
        self.A = A_matrix
        self.line_limits = np.array(line_limits, dtype=np.float64)
        self.num_buses = A_matrix.shape[1]
        self.tolerance = 1e-6

        self.initial_B = initial_B.copy().flatten()

        self.gen_indices = np.where(self.initial_B > self.tolerance)[0]
        self.load_indices = np.where(self.initial_B <= self.tolerance)[0]

        if len(self.gen_indices) == 0: raise ValueError("No generator buses identified.")

        print(f"Identified {len(self.gen_indices)} generator buses (indices: {self.gen_indices})")
        print(f"Identified {len(self.load_indices)} load buses (indices: {self.load_indices}) - Load shedding enabled (Zero Cost).")

       
        flat_gc=np.array(gen_costs).flatten(); flat_gmin=np.array(gen_limits_min).flatten(); flat_gmax=np.array(gen_limits_max).flatten()
        if len(flat_gc)!=self.num_buses or len(flat_gmin)!=self.num_buses or len(flat_gmax)!=self.num_buses: raise ValueError(f"Gen cost/limit array len mismatch")
        self.gen_costs_only=flat_gc[self.gen_indices]; self.gen_limits_min_only=flat_gmin[self.gen_indices]; self.gen_limits_max_only=flat_gmax[self.gen_indices]

        self.initial_B_load_fixed=self.initial_B[self.load_indices] 
        self.load_limits_min_only=self.initial_B_load_fixed
        self.load_limits_max_only=np.zeros_like(self.load_limits_min_only)

        initial_gen_values=self.initial_B[self.gen_indices]; needs_adjust=False
        if np.any(initial_gen_values < self.gen_limits_min_only-self.tolerance) or np.any(initial_gen_values > self.gen_limits_max_only+self.tolerance): print("Warn: Initial gen B outside limits. Clamping..."); self.initial_B[self.gen_indices]=np.clip(initial_gen_values, self.gen_limits_min_only, self.gen_limits_max_only); needs_adjust=True
        req_gen_init=-np.sum(self.initial_B_load_fixed); cur_gen_init=np.sum(self.initial_B[self.gen_indices]); diff_init=req_gen_init-cur_gen_init; num_gens=len(self.gen_indices)
        if abs(diff_init) > self.tolerance*max(1,num_gens): print(f"Warn: Initial sum diff {diff_init:.4f}. Adjusting gens..."); needs_adjust=True
        if num_gens > 0 and abs(diff_init) > self.tolerance*max(1,num_gens): self.initial_B[self.gen_indices]+=diff_init/num_gens; self.initial_B[self.gen_indices]=np.clip(self.initial_B[self.gen_indices], self.gen_limits_min_only, self.gen_limits_max_only)

        initial_gen_values=self.initial_B[self.gen_indices]; needs_adjust=False
        if np.any(initial_gen_values < self.gen_limits_min_only-self.tolerance) or np.any(initial_gen_values > self.gen_limits_max_only+self.tolerance):
            print("Warn: Initial gen B outside limits. Clamping...")
            self.initial_B[self.gen_indices]=np.clip(initial_gen_values, self.gen_limits_min_only, self.gen_limits_max_only)
            needs_adjust=True

        required_total_injection = 0.0
        current_total_injection = np.sum(self.initial_B)
        difference_init = required_total_injection - current_total_injection
        num_gens = len(self.gen_indices)

        if abs(difference_init) > self.tolerance * self.num_buses:
             print(f"Warning: Initial injections sum to {current_total_injection:.4f}. Adjusting generators to balance...")
             if num_gens > 0:
                  diff_per_gen_init = difference_init / num_gens
                  self.initial_B[self.gen_indices] += diff_per_gen_init
                  self.initial_B[self.gen_indices] = np.clip(self.initial_B[self.gen_indices], self.gen_limits_min_only, self.gen_limits_max_only)

                  final_adj_sum = np.sum(self.initial_B)
                 
                  if abs(final_adj_sum) > self.tolerance * self.num_buses * 10:
                       print(f"ERROR: Could not balance initial state even after adjustment. Sum: {final_adj_sum:.6f}")
                 
             else: 
                  print("ERROR: Cannot balance - no generators identified.")
             needs_adjust = True 
        if needs_adjust:
             print("-> Adjusted initial B state used for optimization:", self.initial_B)

       
        self.memory = deque(maxlen=memory_size)

    
    def _apply_constraints(self, solution_vector):
        sol=solution_vector.copy().flatten();
        try: sol[self.load_indices]=np.clip(sol[self.load_indices], self.load_limits_min_only, self.load_limits_max_only)
        except IndexError: pass
        try: sol[self.gen_indices]=np.clip(sol[self.gen_indices], self.gen_limits_min_only, self.gen_limits_max_only)
        except IndexError: pass
        gen_values=sol[self.gen_indices]; load_values=sol[self.load_indices]; num_gens=len(gen_values)
        if num_gens > 0:
            req_gen=-np.sum(load_values); cur_gen=np.sum(gen_values); diff=req_gen-cur_gen
            if abs(diff)>self.tolerance*num_gens: gen_values+=diff/num_gens; sol[self.gen_indices]=np.clip(gen_values, self.gen_limits_min_only, self.gen_limits_max_only)
            final_gen_sum=np.sum(sol[self.gen_indices]); final_diff=req_gen-final_gen_sum
            if abs(final_diff)>self.tolerance*num_gens: gen_values_final=sol[self.gen_indices]; gen_values_final+=final_diff/num_gens; sol[self.gen_indices]=np.clip(gen_values_final, self.gen_limits_min_only, self.gen_limits_max_only)
        return sol.reshape(-1, 1)


    def optimize(self, B_unused, iterations=200, population_size=30, alpha=0.7):
        initial_state_constrained = self._apply_constraints(self.initial_B)
        initial_population = [self._generate_random_solution() for _ in range(population_size)]
        initial_population[0] = initial_state_constrained
        best_solution = initial_state_constrained.copy(); best_fitness = self._calculate_fitness(best_solution)
        fitness_history = [best_fitness] if np.isfinite(best_fitness) else []
        algorithm_contributions = {'OOA': 0, 'KHA': 0, 'SHO': 0}
        for iteration in range(iterations):
            if iteration < iterations*0.3: weights=[0.2,0.6,0.2]
            elif iteration < iterations*0.7: weights=[0.3,0.3,0.4]
            else: weights=[0.6,0.2,0.2]
            new_population = []
            for i in range(population_size):
                algorithm = random.choices(['OOA', 'KHA', 'SHO'], weights=weights, k=1)[0]; current_solution = initial_population[i]
                if algorithm=='OOA': candidate_solution=self._apply_orcas_optimization(initial_population,i,best_solution,iteration,iterations)
                elif algorithm=='KHA': candidate_solution=self._apply_krill_herd(initial_population,i,best_solution,iteration,iterations)
                else: candidate_solution=self._apply_spotted_hyena(initial_population,i,best_solution,iteration,iterations)
                algorithm_contributions[algorithm]+=1
                new_solution=self._apply_constraints(candidate_solution); new_population.append(new_solution)
                fitness=self._calculate_fitness(new_solution) 
                if np.isfinite(fitness) and fitness < best_fitness:
                    best_fitness=fitness; best_solution=new_solution.copy()
                    if iteration % 100 == 0 and i == 0:  
                         deviation = self._get_generator_deviation(best_solution)
                         cost = self._get_rescheduling_cost(best_solution)
                         print(f"Iter {iteration}: Fit={best_fitness:.2f}, Deviation={deviation:.2f}, GenCost={cost:.2f}")
            initial_population = new_population
            if np.isfinite(best_fitness): fitness_history.append(best_fitness)
        best_solution = self._apply_constraints(best_solution)
        if self._is_feasible(best_solution): self._store_in_memory(self.initial_B, best_solution)
        else: print("Warning: Final solution is not feasible.")
        total = sum(algorithm_contributions.values());
        if total > 0: print("\nAlgorithm Contributions:", {k: f"{v/total*100:.1f}%" for k, v in algorithm_contributions.items()})
        final_deviation=self._get_generator_deviation(best_solution); final_gen_cost=self._get_rescheduling_cost(best_solution)
         
        print(f"\nFinal Sum of Absolute Changes (Generators Only): {final_deviation:.4f}"); print(f"Final Generator Rescheduling Cost: {final_gen_cost:.2f} $/hr");
        return best_solution, fitness_history, np.dot(self.A, best_solution)

     
    def _get_generator_deviation(self, solution):
        sol_flat=solution.flatten(); idx=[i for i in self.gen_indices if i<len(sol_flat)]
        if len(idx)!=len(self.gen_indices): return np.inf
        return np.sum(np.abs(sol_flat[idx]-self.initial_B[idx]))
     
    def _get_rescheduling_cost(self, solution):
        sol_flat=solution.flatten(); idx=[i for i in self.gen_indices if i<len(sol_flat)]
        if len(idx)!=len(self.gen_indices) or len(self.gen_costs_only)!=len(idx): return np.inf
        return np.sum(self.gen_costs_only * np.abs(sol_flat[idx]-self.initial_B[idx]))
     

    
    def _calculate_fitness(self, solution):
        """Calculates fitness: Penalties & Deviation(High) + GenCost(Low)."""
        sol_flat=solution.flatten();
        if not np.all(np.isfinite(sol_flat)): return np.inf
        try: C = np.dot(self.A, sol_flat);
        except ValueError: return np.inf
        if not np.all(np.isfinite(C)): return np.inf

         
        penalty_multiplier = 1e10  

         
        line_violations=np.maximum(0, np.abs(C.flatten())-(self.line_limits+self.tolerance));
        line_violation_penalty=penalty_multiplier*np.sum(line_violations**2)

         
        gen_values=sol_flat[self.gen_indices]; gen_limit_penalty=np.inf
        if len(gen_values)==len(self.gen_limits_min_only): violations_lower=np.maximum(0,self.gen_limits_min_only-gen_values); violations_upper=np.maximum(0,gen_values-self.gen_limits_max_only); gen_limit_penalty=penalty_multiplier*(np.sum(violations_lower**2)+np.sum(violations_upper**2))

         load_values=sol_flat[self.load_indices]; load_limit_penalty=np.inf
        if len(load_values)==len(self.load_limits_min_only): violations_lower=np.maximum(0,self.load_limits_min_only-load_values); violations_upper=np.maximum(0,load_values-self.load_limits_max_only); load_limit_penalty=penalty_multiplier*(np.sum(violations_lower**2)+np.sum(violations_upper**2))

         generator_deviation = self._get_generator_deviation(solution)
        deviation_penalty_component = 1e5 * generator_deviation

         gen_rescheduling_cost = self._get_rescheduling_cost(solution)
        gen_cost_weight = 0.1  

         fitness = (line_violation_penalty
                   + gen_limit_penalty
                   + load_limit_penalty
                   + deviation_penalty_component  
                    + gen_cost_weight * gen_rescheduling_cost
                   )

        return fitness if np.isfinite(fitness) else np.inf

     def _is_feasible(self, solution, verbose=False):
        if solution is None:
            if verbose: print("DEBUG (_is_feasible): Input solution is None.")
            return False
        sol_flat = solution.flatten()
        if not np.all(np.isfinite(sol_flat)):
            if verbose: print("DEBUG (_is_feasible): Solution contains non-finite values.")
            return False

         line_ok = False  
        try:
             C=np.dot(self.A,sol_flat); flows=C.flatten()
             if not np.all(np.isfinite(C)): raise ValueError("Flows NaN/Inf")
             line_ok = np.all(np.abs(flows) <= self.line_limits + self.tolerance)  
        except Exception as e:
             line_ok = False;  
             print(f"DEBUG: Line check error: {e}") if verbose else None

         
        gen_values=sol_flat[self.gen_indices]; gen_ok=False  
        if len(gen_values)==len(self.gen_limits_min_only):
             gen_ok=np.all(gen_values>=self.gen_limits_min_only-self.tolerance) and np.all(gen_values<=self.gen_limits_max_only+self.tolerance) 
 
         load_values=sol_flat[self.load_indices]; load_ok=False  
        if len(load_values)==len(self.load_limits_min_only):
             load_ok=np.all(load_values>=self.load_limits_min_only-self.tolerance) and np.all(load_values<=self.load_limits_max_only+self.tolerance)  
         
        bal_ok = abs(np.sum(sol_flat)) < self.tolerance * self.num_buses  

         
        feasible = line_ok and gen_ok and load_ok and bal_ok

         
        if verbose or not feasible:
             print(f"--- Feasibility Check {'FAILED' if not feasible else 'PASSED'} ---")
             if not line_ok:
                  
                 try: flows_debug = np.dot(self.A, sol_flat).flatten()
                 except: flows_debug = np.array([])  
                 failing_lines = np.where(np.abs(flows_debug) > self.line_limits + self.tolerance)[0]
                 print(f"  Line constraints failed (Raw Flow Check). Failing lines: {failing_lines+1}")
                 for idx in failing_lines[:5]: print(f"    Line {idx+1}: Flow={abs(flows_debug[idx]):.6f}, Limit={self.line_limits[idx]:.6f}")
             else: print("  Line constraints met.")
             if not gen_ok:
                  if len(gen_values)==len(self.gen_limits_min_only): 
                     failing_min_g=np.where(gen_values < self.gen_limits_min_only - self.tolerance)[0]; failing_max_g=np.where(gen_values > self.gen_limits_max_only + self.tolerance)[0];
                     if len(failing_min_g)>0: print(f"  Gen Bus {self.gen_indices[failing_min_g[0]]+1} violated MIN Limit ({gen_values[failing_min_g[0]]:.3f} < {self.gen_limits_min_only[failing_min_g[0]]:.3f})")
                     elif len(failing_max_g)>0: print(f"  Gen Bus {self.gen_indices[failing_max_g[0]]+1} violated MAX Limit ({gen_values[failing_max_g[0]]:.3f} > {self.gen_limits_max_only[failing_max_g[0]]:.3f})")
                      
                  else: print("  Generator limits failed (Dim mismatch).")
             else: print("  Generator limits met.")
             if not load_ok:
                  if len(load_values) == len(self.load_limits_min_only):  
                     failing_min_l=np.where(load_values < self.load_limits_min_only - self.tolerance)[0]; failing_max_l=np.where(load_values > self.load_limits_max_only + self.tolerance)[0];
                     if len(failing_min_l)>0: print(f"  Load Bus {self.load_indices[failing_min_l[0]]+1} violated MIN Limit ({load_values[failing_min_l[0]]:.3f} < {self.load_limits_min_only[failing_min_l[0]]:.3f})")
                     elif len(failing_max_l)>0: print(f"  Load Bus {self.load_indices[failing_max_l[0]]+1} violated MAX Limit ({load_values[failing_max_l[0]]:.3f} > {self.load_limits_max_only[failing_max_l[0]]:.3f})")
                      
                  else: print("  Load limits failed (Dim mismatch).")
             else: print("  Load limits met.")
             if not bal_ok: print(f"  Balance check failed, sum={np.sum(sol_flat):.8f}")
             else: print("  Power balance met.")
             print("------------------------------------------")
        return feasible


     
    def _generate_random_solution(self):
        rand_sol = self.initial_B.copy(); n_gens=len(self.gen_indices); n_loads=len(self.load_indices)
        if n_gens>0: g_range=np.maximum(0, self.gen_limits_max_only-self.gen_limits_min_only); pert=(np.random.rand(n_gens)-0.5)*g_range*0.5; rand_sol[self.gen_indices]=np.clip(self.initial_B[self.gen_indices]+pert, self.gen_limits_min_only, self.gen_limits_max_only)
        if n_loads>0: ld_range=0.0-self.load_limits_min_only; pert=np.random.rand(n_loads)*ld_range*0.1; rand_sol[self.load_indices]=self.initial_B[self.load_indices]+pert
        return self._apply_constraints(rand_sol)

     
    def _check_memory(self, B_unused):
        if not self.memory: return None
        for _, stored_B in reversed(self.memory):
            if self._is_feasible(stored_B): print("Using feasible solution from memory."); return stored_B
        return None
    def _store_in_memory(self, iB, oB): self.memory.append((iB.copy(), oB.copy())); print(f"Feasible solution stored. Mem size: {len(self.memory)}")

     
    def _apply_orcas_optimization(self,p,i,b,it,m_it): 
        s=p[i].copy();a=2*(1-it/m_it);r1,r2=random.random(),random.random();
        if r1<0.5: d=np.abs(b-s);l=2*r2-1;n=d*np.exp(1*l)*np.cos(2*np.pi*l)+b
        else: idx=[j for j in range(len(p)) if j!=i];r=random.choice(idx) if idx else i;X=p[r];A=2*a*r1-a;C=2*r2;D=np.abs(C*X-s);n=X-A*D
        return np.nan_to_num(n,nan=0.,posinf=1e6,neginf=-1e6)
    def _apply_krill_herd(self,p,i,b,it,m_it): s=p[i].copy();Vf=.02;Nmax=.01;Dt=1;Ni=Nmax*(b-s);Fi=Vf*(b-s);d=np.random.uniform(-1,1,s.shape);Dmax=.005*(1-it/m_it);Di=Dmax*d; n=s+Dt*(Ni+Fi+Di);return np.nan_to_num(n,nan=0.,posinf=1e6,neginf=-1e6)
    def _apply_spotted_hyena(self,p,i,b,it,m_it): 
        s=p[i].copy();h=5-it*(5/m_it);B=2*random.random();E=2*h*random.random()-h;D_b=np.abs(B*b-s);X1=b-E*D_b;
        if abs(E)>=1: idx=[j for j in range(len(p)) if j!=i];r_h=p[random.choice(idx)] if idx else s;D_h=np.abs(B*r_h-s);n=r_h-E*D_h
        else: n=X1
        return np.nan_to_num(n,nan=0.,posinf=1e6,neginf=-1e6)

In [ ]:
def optimize_power_flow_free_loadshed(A, B, line_limits, gen_costs, gen_limits_min, gen_limits_max, iterations=5000, population_size=100):
    """
    Wrapper function. Objective: 1. Constraints & Min Deviation, 2. Min Gen Cost.
    Allows load shedding but assigns no cost to it.
    """
    print("--- Initializing Optimizer (Load Shedding Enabled - Zero Cost) ---")
    try:
           
        optimizer = HybridPowerFlowOptimizer(A, line_limits, gen_costs, gen_limits_min, gen_limits_max, B)
    except ValueError as e:
        print(f"ERROR initializing optimizer: {e}"); return B, [], np.dot(A, B), np.dot(A, B), False, {}

    C_unoptimized = np.dot(A, B)

    print("\n--- Checking Initial State Feasibility ---")
    initial_state = optimizer.initial_B.reshape(-1, 1); initial_feasible = optimizer._is_feasible(initial_state, verbose=True)
    print(f"Initial state feasible: {initial_feasible}"); print("WARNING: Starting infeasible.") if not initial_feasible else None

    print("\n--- Starting Optimization (Load Shedding Enabled - Zero Cost) ---")
    B_opt, fit_hist, C_opt = optimizer.optimize(None, iterations, population_size=population_size, alpha=0.7)

    print("\n--- Checking Final Solution Feasibility ---")
    final_feasible = optimizer._is_feasible(B_opt, verbose=True)
    print(f"\nFinal feasibility: {final_feasible}")
    details = {"gen_indices": optimizer.gen_indices, "load_indices": optimizer.load_indices}
    return B_opt, fit_hist, C_opt, C_unoptimized, final_feasible, details

In [ ]:
 def visualize_results(A, B, B_optimized, C_optimized, C_unoptimized, line_limits, gen_costs, gen_limits_min, gen_limits_max, gen_indices, load_indices, fitness_history):
    num_lines=A.shape[0]; num_buses=A.shape[1]; tolerance=1e-6
    print("\n--- Generating Plots (Load Shedding Enabled - Zero Cost) ---")
     
    try:  
        plt.figure(figsize=(12,6)); idx=np.arange(1,num_lines+1); plt.plot(idx,C_unoptimized.flatten(),'o-',label='Unoptimized',alpha=.7); plt.plot(idx,C_optimized.flatten(),'s--',label='Optimized',alpha=.9); plt.plot(idx,line_limits,'r:',alpha=.8,label='Limit(+)'); plt.plot(idx,-line_limits,'r:',alpha=.8,label='Limit(-)')
        flows_opt_abs=np.abs(C_optimized.flatten()); violations=np.where(flows_opt_abs>line_limits+tolerance)[0]
        if len(violations)>0: plt.scatter(idx[violations],C_optimized.flatten()[violations],c='m',s=100,zorder=5,label='Violations')
        plt.xlabel('Line');plt.ylabel('Flow');plt.title('Line Flows (Load Shed Allowed - Zero Cost)');plt.legend();plt.grid(True);plt.xticks(idx);plt.tight_layout();plt.show()
    except Exception as e: print(f"Plot error (Lines): {e}")
    try:  
        if fitness_history and len(fitness_history)>1: plt.figure(figsize=(10,5));plt.plot(fitness_history,'.-b',label='Best Fitness');plt.xlabel('Iteration');plt.ylabel('Fitness(Log)');plt.title('Convergence');plt.yscale('log');plt.legend();plt.grid(True);plt.tight_layout();plt.show()
    except Exception as e: print(f"Plot error (Fitness): {e}")
    try: 
        plt.figure(figsize=(14,7)); bus_idx_plot=np.arange(1,num_buses+1); bw=.35; colors_i=['blue' if i in gen_indices else 'lightblue' for i in range(num_buses)]; colors_o=['green' if i in gen_indices else 'lightgreen' for i in range(num_buses)]
        plt.bar(bus_idx_plot-bw/2,B.flatten(),width=bw,label='Initial(Gen=dk)',alpha=.7,color=colors_i); plt.bar(bus_idx_plot+bw/2,B_optimized.flatten(),width=bw,label='Optimized(Gen=dk)',alpha=.7,color=colors_o)
        if len(gen_indices)>0: plt.scatter(bus_idx_plot[gen_indices],gen_limits_max[gen_indices],c='grey',marker='_',s=100,label='Gen Max'); plt.scatter(bus_idx_plot[gen_indices],gen_limits_min[gen_indices],c='grey',marker='_',s=100,label='Gen Min')
        if len(load_indices)>0: plt.scatter(bus_idx_plot[load_indices],np.zeros(len(load_indices)),c='pink',marker='_',s=100,label='Load Max(0)'); plt.scatter(bus_idx_plot[load_indices],B.flatten()[load_indices],c='pink',marker='_',s=100,label='Load Min(Initial)')
        plt.xlabel('Bus');plt.ylabel('Injection');plt.title('Injections');plt.xticks(bus_idx_plot);plt.legend();plt.grid(True,axis='y');plt.tight_layout();plt.show()
    except Exception as e: print(f"Plot error (Injections): {e}")
    try:  
        changes=B_optimized.flatten()-B.flatten()
        if len(gen_indices)>0: plt.figure(figsize=(10,4));g_chg=changes[gen_indices];g_lbl=bus_idx_plot[gen_indices];colors_g=['g' if x>=0 else 'r' for x in g_chg];bar_idx_g=np.arange(len(g_indices));plt.bar(bar_idx_g,g_chg,color=colors_g);plt.axhline(0,c='k',ls='-');plt.xlabel('Gen Bus');plt.ylabel('Change');plt.title('Gen Changes');plt.xticks(bar_idx_g,g_lbl);plt.grid(True,axis='y');plt.tight_layout();plt.show()
        if len(load_indices)>0: plt.figure(figsize=(10,4));l_chg=changes[load_indices];l_lbl=bus_idx_plot[load_indices];colors_l=['orange' if x>tolerance else 'grey' for x in l_chg];bar_idx_l=np.arange(len(load_indices));plt.bar(bar_idx_l,l_chg,color=colors_l);plt.axhline(0,c='k',ls='-');plt.xlabel('Load Bus');plt.ylabel('Change(Shed)');plt.title('Load Changes(Shedding)');plt.xticks(bar_idx_l,l_lbl);plt.grid(True,axis='y');plt.tight_layout();plt.show()
    except Exception as e: print(f"Plot error (Changes): {e}")

    print("\n"+"="*30+" RESULTS SUMMARY (Load Shed Allowed - Zero Cost) "+"="*30); np.set_printoptions(precision=2, suppress=True)
    try:
        final_dev_g=np.sum(np.abs(B_optimized.flatten()[gen_indices]-B.flatten()[gen_indices]))
        g_costs_only=np.array(gen_costs).flatten()[gen_indices]; final_g_cost=np.sum(g_costs_only * np.abs(B_optimized.flatten()[gen_indices]-B.flatten()[gen_indices]))
         
        ls_amount=np.maximum(0, B_optimized.flatten()[load_indices]-B.flatten()[load_indices]); total_ls=np.sum(ls_amount)
        print(f"\nInitial B:\n{B.flatten()}"); print(f"\nOptimized B:\n{B_optimized.flatten()}"); print(f"Sum Opt B:{np.sum(B_optimized):.6f}")
        print(f"\nInitial Flows:\n{C_unoptimized.flatten()}"); print(f"\nOptimized Flows:\n{C_optimized.flatten()}"); print("-"*70)
        print("\nObjective Metrics & Load Shed:"); print(f"  Gen Deviation: {final_dev_g:.4f}"); print(f"  Gen Cost: {final_g_cost:.2f}"); print(f"  Total Load Shed: {total_ls:.2f} MW"); 
    print("\nConstraint Check Summary:");
    try:  
         dummy_gc=np.zeros(num_buses); dummy_lsc=np.zeros(num_buses); dummy_min=np.zeros(num_buses); dummy_max=np.zeros(num_buses)
         if len(gen_indices)>0: dummy_min[gen_indices]=gen_limits_min[gen_indices]; dummy_max[gen_indices]=gen_limits_max[gen_indices]
         if len(load_indices)>0: dummy_min[load_indices]=B.flatten()[load_indices]-1e-9; dummy_max[load_indices]=0.0+1e-9
          
         temp_opt = HybridPowerFlowOptimizer(A, line_limits, dummy_gc, dummy_min, dummy_max, dummy_lsc, B)
         is_final_feasible = temp_opt._is_feasible(B_optimized)
         print(f"  Final solution feasible: {'YES' if is_final_feasible else 'NO'}")
    except Exception as e: print(f"Error in final feas check: {e}")
    print("="*70+"\nNote: Voltage constraints not included.\n"+"="*70)

In [ ]:
if __name__ == "__main__":

     
    A = np.array([
        [0.5857,-0.2571,-0.0428,-0.0857,-0.2000],
        [0.2143,0.0571,-0.1572,-0.1143,0.0000],
        [0.0905,0.1619,-0.1953,-0.1238,0.0666],
        [0.1080,0.1651,-0.1206,-0.1968,0.0444],
        [0.1872,0.2158,0.0730,0.0349,-0.5109],
        [0.1048,0.0191,0.4476,-0.4380,-0.1334],
        [0.0128,-0.0158,0.1270,0.1651,-0.2891]
    ])
    num_lines_main = A.shape[0]; num_buses_main = A.shape[1]

     
    while True:
         
        print("\n" + "="*25 + " New Scenario (Load Shed Allowed - Zero Cost) " + "="*25)
        print("Objective: 1. Meet Limits & Min Gen Deviation, 2. Min Gen Cost")

        
        try:
            print(f"\n>>> Enter LINE power limits ({num_lines_main} values):")
             
            line_limits_input = np.abs(np.array([float(input(f"  Line {i+1} limit: ")) for i in range(num_lines_main)]))

            print(f"\n>>> Enter INITIAL power injections (B) for ALL buses ({num_buses_main} values):")
            print("    (Positive for Generators, Negative for Loads)")
             
            B_input_list = [float(input(f"  Bus {i+1} injection: ")) for i in range(num_buses_main)]; B_input=np.array(B_input_list).reshape(-1,1)
            temp_gen_indices = np.where(B_input.flatten() > 1e-6)[0]; temp_load_indices = np.where(B_input.flatten() <= 1e-6)[0]
            if len(temp_gen_indices) == 0: print("ERROR: No Gens!"); continue; print(f"-> Gens: {temp_gen_indices+1}, Loads: {temp_load_indices+1}")

            gen_costs_input = np.zeros(num_buses_main); gen_limits_min_input = np.zeros(num_buses_main); gen_limits_max_input = np.zeros(num_buses_main)
            print(f"\n>>> Enter GENERATOR specific data ONLY for buses {temp_gen_indices+1}:")

             
            print("  Enter Gen Cost Coefficients ($/MWh):")
             
            for i in temp_gen_indices:
                 
                 gen_costs_input[i] = float(input(f"    G{i+1} cost: "))

            print("  Enter Gen MIN Limit (MW):")
             
            for i in temp_gen_indices:
                  
                 gen_limits_min_input[i] = float(input(f"    G{i+1} min: "))

            print("  Enter Gen MAX Limit (MW):")
             
            for i in temp_gen_indices:
                 
                 gen_limits_max_input[i] = float(input(f"    G{i+1} max: "))
             

            if np.any(gen_limits_max_input[temp_gen_indices] < gen_limits_min_input[temp_gen_indices]): print("ERROR: Gen max<min."); continue

             
 
        except ValueError: print("Invalid input."); continue

         
        print("\n" + "="*25 + " Running Optimization (Load Shed Allowed - Zero Cost) " + "="*25)
         
        B_optimized, fitness_history, C_optimized, C_unoptimized, final_feasible, opt_details = optimize_power_flow_free_loadshed(
            A, B_input, line_limits_input, gen_costs_input, gen_limits_min_input, gen_limits_max_input,
            iterations=5000,  
            population_size=100  
        )

        print("\n" + "="*25 + " Optimization Finished " + "="*25)

         
        viz_gen_indices = opt_details.get("gen_indices", np.array([])); viz_load_indices = opt_details.get("load_indices", np.array([]))
        print("\nVisualizing results...")
         
        visualize_results(A, B_input, B_optimized, C_optimized, C_unoptimized, line_limits_input,
                          gen_costs_input, gen_limits_min_input, gen_limits_max_input,
                          viz_gen_indices, viz_load_indices,
                          fitness_history)

         
        if final_feasible:
            save = input("\nSave detailed results to file? (y/n): ").strip().lower()
            if save == 'y':
                fname = input("Enter filename (default: power_flow_free_loadshed_results.txt): ").strip() or "power_flow_free_loadshed_results.txt"
                try:
                    with open(fname, 'w') as f:
                        
                        gen_costs_only_save=gen_costs_input[viz_gen_indices]; final_gen_cost=np.sum(gen_costs_only_save*np.abs(B_optimized.flatten()[viz_gen_indices]-B_input.flatten()[viz_gen_indices]))
                        final_deviation_gens=np.sum(np.abs(B_optimized.flatten()[viz_gen_indices]-B_input.flatten()[viz_gen_indices]))
                        load_shed_amount=np.maximum(0, B_optimized.flatten()[viz_load_indices]-B_input.flatten()[viz_load_indices]); total_load_shed=np.sum(load_shed_amount)

                        f.write("Power Flow Opt Results (Load Shed Allowed - Zero Cost)\n"); f.write("Objective: 1. Limits & Min Deviation, 2. Min Gen Cost\n"); f.write("="*30+"\n\n")
                        f.write(f"Matrix A:\n"); np.savetxt(f, A, fmt='%.4f'); f.write("\n")
                        f.write("Line Limits:\n"); [f.write(f"L{i+1}: {l:.2f}\n") for i,l in enumerate(line_limits_input)]
                        f.write("\nGen Costs:\n"); [f.write(f"G{viz_gen_indices[i]+1}: {c:.2f}\n") for i,c in enumerate(gen_costs_only_save)]
                        f.write("\nGen Min Limits:\n"); [f.write(f"G{idx+1}: {gen_limits_min_input[idx]:.2f}\n") for idx in viz_gen_indices]
                        f.write("\nGen Max Limits:\n"); [f.write(f"G{idx+1}: {gen_limits_max_input[idx]:.2f}\n") for idx in viz_gen_indices]
                         
                        f.write("\nInitial B:\n"); [f.write(f"Bus {i+1}: {b[0]:.2f}\n") for i,b in enumerate(B_input)]
                        f.write("\nOptimized B:\n"); [f.write(f"Bus {i+1}: {b[0]:.2f}\n") for i,b in enumerate(B_optimized)]
                        f.write("\nInitial C:\n"); [f.write(f"L{i+1}: {c[0]:.2f}\n") for i,c in enumerate(C_unoptimized)]
                        f.write("\nOptimized C:\n"); [f.write(f"L{i+1}: {c[0]:.2f}\n") for i,c in enumerate(C_optimized)]
                        f.write(f"\nFinal Gen Deviation: {final_deviation_gens:.4f}\n"); f.write(f"Final Gen Cost: {final_gen_cost:.2f}\n"); f.write(f"Total Load Shed: {total_load_shed:.2f} MW\n");
                        f.write(f"\nFeasible: {'YES' if final_feasible else 'NO'}\n")
                    print(f"Results saved to {fname}")
                except IOError as e: print(f"Error saving results: {e}")
        elif not final_feasible: print("\nFinal solution was infeasible.")

         
        if input("\nRun another scenario? (y/n):").strip().lower()!='y': print("\nExiting..."); break
        else: print("\nRestarting...\n")